# ECG Comprehensive Comparison Analysis
## Threshold-Based Arrhythmia Detection — No ML Required

| Record | Condition |
|--------|----------|
| **100** | Normal Sinus Rhythm |
| **106** | Ventricular Arrhythmia |
| **207** | Atrial Fibrillation |

**Five signal processing methods** are used to extract diagnostic parameters, each compared against clinical thresholds.  
A **majority voting rule (≥ 3/5 methods)** determines the final diagnosis.

---
## Cell 1 — Imports & Setup

In [ ]:
import wfdb
import numpy as np
from scipy import signal
from scipy.signal import find_peaks
import pywt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'text.color':       'white',
    'grid.color':       '#555',
    'grid.alpha':       0.3,
})

CASES  = [('100', 'Normal Sinus Rhythm'), ('106', 'Ventricular Arrhythmia'), ('207', 'Atrial Fibrillation')]
COLORS = {'100': '#2ecc71', '106': '#e74c3c', '207': '#e67e22'}
print('✅ Setup complete!')

---
## Cell 2 — Signal Processing Functions

In [ ]:
# ── Load & Preprocess ──────────────────────────────────────────
def load_ecg(record_name):
    record = wfdb.rdrecord(record_name, pn_dir='mitdb')
    ecg = record.p_signal[:10000, 0]
    return ecg, record.fs

def preprocess_ecg(ecg, fs):
    sos = signal.butter(4, 0.5, btype='high', fs=fs, output='sos')
    ecg = signal.sosfiltfilt(sos, ecg)
    b, a = signal.iirnotch(60, 30, fs)
    return signal.filtfilt(b, a, ecg)

def detect_r_peaks(ecg, fs):
    diff2 = np.diff(ecg) ** 2
    win = int(0.15 * fs)
    integrated = np.convolve(diff2, np.ones(win)/win, mode='same')
    peaks, _ = find_peaks(integrated, distance=int(0.2*fs), height=np.mean(integrated))
    return peaks

# ── Method 1: Histogram ────────────────────────────────────────
def histogram_analysis(ecg, fs):
    r_peaks = detect_r_peaks(ecg, fs)
    rr = np.diff(r_peaks) / fs * 1000
    p = {'mean_rr': np.mean(rr), 'std_rr': np.std(rr),
         'cv_rr': np.std(rr)/np.mean(rr), 'min_rr': np.min(rr),
         'max_rr': np.max(rr), 'range_rr': np.max(rr)-np.min(rr),
         'mean_hr': 60000/np.mean(rr)}
    flags = {'std_rr > 100ms': p['std_rr'] > 100,
             'CV > 0.15':      p['cv_rr'] > 0.15,
             'range > 400ms':  p['range_rr'] > 400}
    return rr, p, flags, any(flags.values())

# ── Method 2: Wavelet ──────────────────────────────────────────
def wavelet_analysis(ecg, fs):
    coeffs = pywt.wavedec(ecg, 'db4', level=5)
    energies = [np.sum(c**2) for c in coeffs]
    ep = [e/sum(energies)*100 for e in energies]
    p = {'energy_distribution': ep, 'detail_energy': sum(ep[1:]),
         'approx_energy': ep[0], 'dominant_level': int(np.argmax(ep))}
    flags = {'detail_energy > 60%': p['detail_energy'] > 60,
             'dominant level > 2':  p['dominant_level'] > 2}
    return coeffs, p, flags, any(flags.values())

# ── Method 3: STFT ─────────────────────────────────────────────
def stft_analysis(ecg, fs):
    f, t, Zxx = signal.stft(ecg, fs, nperseg=256, noverlap=250)
    spec = np.abs(Zxx)**2
    fmask = f <= 10
    dom_freqs = f[fmask][np.argmax(spec[fmask, :], axis=0)]
    mdf = np.mean(dom_freqs)
    p = {'mean_dominant_freq': mdf, 'std_dominant_freq': np.std(dom_freqs),
         'freq_stability_cv': np.std(dom_freqs)/mdf if mdf > 0 else 0,
         'freq_range': np.max(dom_freqs)-np.min(dom_freqs)}
    flags = {'std > 0.3 Hz': p['std_dominant_freq'] > 0.3,
             'CV > 0.2':     p['freq_stability_cv'] > 0.2,
             'range > 1.5Hz':p['freq_range'] > 1.5}
    return f, t, Zxx, p, flags, any(flags.values())

# ── Method 4: FFT ──────────────────────────────────────────────
def fft_analysis(ecg, fs):
    N = len(ecg)
    fft_vals = np.fft.fft(ecg)
    freqs = np.fft.fftfreq(N, 1/fs)
    pos = freqs > 0
    freqs, power = freqs[pos], np.abs(fft_vals[pos])**2
    cm = (freqs >= 0.5) & (freqs <= 5)
    cf, cp = freqs[cm], power[cm]
    pks, _ = find_peaks(cp, height=np.max(cp)*0.1)
    dom = cf[np.argmax(cp)]
    p = {'num_peaks': len(pks), 'dominant_freq': dom,
         'dominant_hr': dom*60, 'peak_frequencies': cf[pks],
         'spectral_energy': np.sum(cp)}
    flags = {'peaks > 3':        p['num_peaks'] > 3,
             'HR outside 50-120':p['dominant_hr'] < 50 or p['dominant_hr'] > 120,
             'no clear peak':    p['num_peaks'] == 0}
    return freqs, power, p, flags, any(flags.values())

# ── Method 5: Entropy ──────────────────────────────────────────
def entropy_analysis(ecg, fs):
    sig = ecg[:2000]
    hist, _ = np.histogram(sig, bins=50, density=True)
    hist = hist[hist > 0]
    shannon = -np.sum(hist * np.log2(hist))

    def apen(s, m=2):
        r = 0.2*np.std(s); N = len(s)
        def phi(m):
            pats = np.array([s[i:i+m] for i in range(N-m+1)])
            C = [np.sum(np.max(np.abs(pats-pats[i]),axis=1)<=r)/(N-m+1.0) for i in range(len(pats))]
            return np.mean(np.log(C))
        return abs(phi(m+1)-phi(m))

    def sampen(s, m=2):
        r = 0.2*np.std(s); N = len(s)
        def phi(m):
            pats = np.array([s[i:i+m] for i in range(N-m)])
            B = 0
            for i in range(len(pats)):
                d = np.max(np.abs(pats-pats[i]), axis=1)
                B += np.sum((d<=r) & (np.arange(len(pats))!=i))
            return B
        A, B = phi(m+1), phi(m)
        return -np.log(A/B) if B!=0 and A!=0 else 0

    try:
        ap = apen(sig)
        sp = sampen(sig)
    except:
        ap = sp = 0

    p = {'shannon_entropy': shannon, 'approximate_entropy': ap, 'sample_entropy': sp}
    flags = {'ApEn > 1.5':   ap > 1.5,
             'SampEn > 1.2': sp > 1.2}
    return p, flags, any(flags.values())

print('✅ All analysis functions defined!')

---
## Cell 3 — Run Analysis on All 3 Records

In [ ]:
all_results = {}

for record_id, label in CASES:
    print(f"\n{'='*60}")
    print(f"  ANALYZING Record {record_id}: {label}")
    print(f"{'='*60}")

    ecg_raw, fs = load_ecg(record_id)
    ecg = preprocess_ecg(ecg_raw, fs)

    print("  [1/5] Histogram Analysis...", end=' ')
    rr, h_p, h_f, h_abn = histogram_analysis(ecg, fs)
    print(f"{'❌ ABNORMAL' if h_abn else '✅ NORMAL'}")

    print("  [2/5] Wavelet Analysis...  ", end=' ')
    coeffs, w_p, w_f, w_abn = wavelet_analysis(ecg, fs)
    print(f"{'❌ ABNORMAL' if w_abn else '✅ NORMAL'}")

    print("  [3/5] STFT Analysis...     ", end=' ')
    f_arr, t_arr, Zxx, s_p, s_f, s_abn = stft_analysis(ecg, fs)
    print(f"{'❌ ABNORMAL' if s_abn else '✅ NORMAL'}")

    print("  [4/5] FFT Analysis...      ", end=' ')
    freqs, power, ff_p, ff_f, ff_abn = fft_analysis(ecg, fs)
    print(f"{'❌ ABNORMAL' if ff_abn else '✅ NORMAL'}")

    print("  [5/5] Entropy Analysis...  ", end=' ')
    e_p, e_f, e_abn = entropy_analysis(ecg, fs)
    print(f"{'❌ ABNORMAL' if e_abn else '✅ NORMAL'}")

    abnormal_count = sum([h_abn, w_abn, s_abn, ff_abn, e_abn])
    final_diag = "⚠️  ARRHYTHMIA DETECTED" if abnormal_count >= 3 else "✓  NORMAL SINUS RHYTHM"

    print(f"\n  Votes: {abnormal_count}/5 methods flagged abnormal")
    print(f"  FINAL DIAGNOSIS ➜ {final_diag}")

    all_results[record_id] = {
        'label': label, 'ecg': ecg, 'fs': fs, 'rr': rr,
        'Zxx': Zxx, 'f_stft': f_arr, 't_stft': t_arr,
        'freqs': freqs, 'power': power,
        'hist':    {'params': h_p,  'flags': h_f,  'abnormal': h_abn},
        'wavelet': {'params': w_p,  'flags': w_f,  'abnormal': w_abn},
        'stft':    {'params': s_p,  'flags': s_f,  'abnormal': s_abn},
        'fft':     {'params': ff_p, 'flags': ff_f, 'abnormal': ff_abn},
        'entropy': {'params': e_p,  'flags': e_f,  'abnormal': e_abn},
        'abnormal_count': abnormal_count,
        'final_diagnosis': final_diag,
    }

print(f"\n{'='*60}")
print("  ALL 3 RECORDS PROCESSED SUCCESSFULLY")
print(f"{'='*60}")

---
## Cell 4 — Comparison Table (All Parameters)

In [ ]:
records = ['100', '106', '207']

def print_table_row(name, vals, fmt="{:.4f}", threshold=None, higher_is_bad=True):
    cells = []
    for v in vals:
        s = fmt.format(v)
        if threshold is not None:
            flag = (v > threshold) if higher_is_bad else (v < threshold)
            s = ("[!!] " if flag else "[ ]  ") + s
        cells.append(s.rjust(22))
    print(f"{name:<32}" + "".join(cells))

hdr = f"{'Parameter':<32}" + "".join(f"{'R'+r+' ('+all_results[r]['label'][:10]+')':>22}" for r in records)
div = "-" * (32 + 22*3)

print("\n" + "="*78)
print(" COMPREHENSIVE PARAMETER COMPARISON TABLE")
print("="*78)
print("  [!!] = Threshold exceeded (ABNORMAL)  |  [ ] = Within range (NORMAL)")
print("="*78)
print(hdr)

print(div)
print("\n▶ METHOD 1: HISTOGRAM (RR Interval Variability)")
print("  Thresholds: std_rr > 100 ms, CV > 0.15, range > 400 ms")
print(div)
print_table_row("  Mean RR (ms)",   [all_results[r]['hist']['params']['mean_rr']   for r in records])
print_table_row("  Std RR (ms)   [thresh: >100]", [all_results[r]['hist']['params']['std_rr']    for r in records], threshold=100)
print_table_row("  CV (Coeff Var) [thresh: >0.15]", [all_results[r]['hist']['params']['cv_rr']     for r in records], threshold=0.15)
print_table_row("  Range RR (ms) [thresh: >400]", [all_results[r]['hist']['params']['range_rr']  for r in records], threshold=400)
print_table_row("  Mean HR (bpm)",  [all_results[r]['hist']['params']['mean_hr']   for r in records])

print(div)
print("\n▶ METHOD 2: WAVELET (Energy Distribution)")
print("  Threshold: detail_energy > 60%")
print(div)
print_table_row("  Approx Energy A5 (%)", [all_results[r]['wavelet']['params']['approx_energy']  for r in records])
print_table_row("  Detail Energy    [thresh: >60%]", [all_results[r]['wavelet']['params']['detail_energy'] for r in records], threshold=60)
print_table_row("  Dominant Level  [thresh: >2]", [all_results[r]['wavelet']['params']['dominant_level'] for r in records], fmt="{:.0f}", threshold=2)

print(div)
print("\n▶ METHOD 3: STFT (Spectral Stability)")
print("  Threshold: freq_stability_cv > 0.2")
print(div)
print_table_row("  Mean Dominant Freq (Hz)", [all_results[r]['stft']['params']['mean_dominant_freq'] for r in records])
print_table_row("  Std Dom Freq (Hz) [thresh: >0.3]", [all_results[r]['stft']['params']['std_dominant_freq']  for r in records], threshold=0.3)
print_table_row("  Freq Stability CV [thresh: >0.2]", [all_results[r]['stft']['params']['freq_stability_cv']  for r in records], threshold=0.2)
print_table_row("  Freq Range (Hz)   [thresh: >1.5]", [all_results[r]['stft']['params']['freq_range']         for r in records], threshold=1.5)

print(div)
print("\n▶ METHOD 4: FFT (Power Spectrum)")
print("  Thresholds: peaks > 3, HR outside 50-120 bpm")
print(div)
print_table_row("  Dominant HR (bpm)", [all_results[r]['fft']['params']['dominant_hr']  for r in records])
print_table_row("  # of Freq Peaks  [thresh: >3]", [all_results[r]['fft']['params']['num_peaks']     for r in records], fmt="{:.0f}", threshold=3)

print(div)
print("\n▶ METHOD 5: ENTROPY (Signal Complexity)")
print("  Thresholds: ApEn > 1.5, SampEn > 1.2")
print(div)
print_table_row("  Shannon Entropy",           [all_results[r]['entropy']['params']['shannon_entropy']      for r in records])
print_table_row("  Approx Entropy  [thresh: >1.5]", [all_results[r]['entropy']['params']['approximate_entropy'] for r in records], threshold=1.5)
print_table_row("  Sample Entropy  [thresh: >1.2]", [all_results[r]['entropy']['params']['sample_entropy']       for r in records], threshold=1.2)

print("\n" + "="*78)

---
## Cell 5 — Voting Diagnosis Summary

In [ ]:
methods      = ['hist', 'wavelet', 'stft', 'fft', 'entropy']
method_names = ['Histogram', 'Wavelet', 'STFT', 'FFT', 'Entropy']

print("\n" + "="*78)
print(" THRESHOLD-BASED DIAGNOSIS VOTING SUMMARY")
print("="*78)

# Header
print(f"\n{'Method':<14}", end='')
for r in records:
    lbl = f"R{r} ({all_results[r]['label'][:12]})"
    print(f"  {lbl:<24}", end='')
print()
print("-"*78)

# Each method row
for m, mn in zip(methods, method_names):
    print(f"{mn:<14}", end='')
    for r in records:
        abn = all_results[r][m]['abnormal']
        tag = "❌  ABNORMAL" if abn else "✅  NORMAL  "
        print(f"  {tag:<24}", end='')
    print()

print("-"*78)

# Vote counts
print(f"{'Vote Count':<14}", end='')
for r in records:
    cnt = all_results[r]['abnormal_count']
    bar = '█' * cnt + '░' * (5 - cnt)
    print(f"  {str(cnt)+'/5  '+bar:<24}", end='')
print()

print("="*78)
print(f"{'DIAGNOSIS':<14}", end='')
for r in records:
    diag = all_results[r]['final_diagnosis']
    print(f"  {diag[:24]:<24}", end='')
print()
print("="*78)

print("\nDiagnosis Rule: ≥ 3/5 methods flagged as abnormal  ➜  ARRHYTHMIA DETECTED")
print("                < 3/5 methods flagged as abnormal  ➜  NORMAL SINUS RHYTHM")

---
## Cell 6 — Visualization: ECG Signals + Key Parameter Bar Charts

In [ ]:
fig = plt.figure(figsize=(22, 30))
fig.patch.set_facecolor('#1a1a2e')
gs = GridSpec(6, 3, figure=fig, hspace=0.60, wspace=0.38)

SHORT_LABELS = {r: all_results[r]['label'] for r in records}

def styled_bar(ax, title, val_dict, ylabel, threshold=None, thresh_label=None):
    recs = list(val_dict.keys())
    vals = list(val_dict.values())
    colors = [COLORS[r] for r in recs]
    bars = ax.bar([SHORT_LABELS[r][:13] for r in recs], vals,
                  color=colors, width=0.5, edgecolor='white', linewidth=0.6)
    if threshold is not None:
        ax.axhline(threshold, color='yellow', linestyle='--', linewidth=1.5,
                   label=f'Threshold: {threshold}' + (f' ({thresh_label})' if thresh_label else ''))
        ax.legend(fontsize=7.5, facecolor='#222', labelcolor='yellow')
    ax.set_title(title, fontsize=9.5, fontweight='bold', pad=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.tick_params(axis='x', labelsize=8, rotation=12)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8, color='white', fontweight='bold')

# ── ROW 0: Raw ECG signals ──
for i, r in enumerate(records):
    ax = fig.add_subplot(gs[0, i])
    ecg = all_results[r]['ecg']
    fs  = all_results[r]['fs']
    t   = np.arange(1800) / fs
    ax.plot(t, ecg[:1800], color=COLORS[r], linewidth=0.9)
    diag = all_results[r]['final_diagnosis']
    ax.set_title(f"Record {r} — {SHORT_LABELS[r]}\n→ {diag}",
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=7)
    ax.set_ylabel('Amplitude (mV)', fontsize=7)
    ax.grid(True, alpha=0.2)

# ── ROW 1: Histogram ──
ax = fig.add_subplot(gs[1, 0])
styled_bar(ax, "Std RR Interval (ms)\n[Abnormal if > 100ms]",
           {r: all_results[r]['hist']['params']['std_rr'] for r in records},
           "Std RR (ms)", threshold=100)

ax = fig.add_subplot(gs[1, 1])
styled_bar(ax, "RR Coeff. of Variation\n[Abnormal if > 0.15]",
           {r: all_results[r]['hist']['params']['cv_rr'] for r in records},
           "CV", threshold=0.15)

ax = fig.add_subplot(gs[1, 2])
styled_bar(ax, "RR Range (ms)\n[Abnormal if > 400ms]",
           {r: all_results[r]['hist']['params']['range_rr'] for r in records},
           "Range (ms)", threshold=400)

# ── ROW 2: Wavelet + STFT ──
ax = fig.add_subplot(gs[2, 0])
styled_bar(ax, "Wavelet Detail Energy (%)\n[Abnormal if > 60%]",
           {r: all_results[r]['wavelet']['params']['detail_energy'] for r in records},
           "Detail Energy (%)", threshold=60)

ax = fig.add_subplot(gs[2, 1])
styled_bar(ax, "STFT Freq Stability CV\n[Abnormal if > 0.2]",
           {r: all_results[r]['stft']['params']['freq_stability_cv'] for r in records},
           "CV", threshold=0.2)

ax = fig.add_subplot(gs[2, 2])
ax2 = fig.add_subplot(gs[2, 2])
styled_bar(ax2, "FFT Dominant Heart Rate (bpm)\n[Normal zone: 50–120 bpm]",
           {r: all_results[r]['fft']['params']['dominant_hr'] for r in records},
           "HR (bpm)")
ax2.axhspan(50, 120, alpha=0.15, color='lime', label='Normal Zone (50-120 bpm)')
ax2.legend(fontsize=7.5, facecolor='#222', labelcolor='lime')

# ── ROW 3: Entropy ──
ax = fig.add_subplot(gs[3, 0])
styled_bar(ax, "Approximate Entropy\n[Abnormal if > 1.5]",
           {r: all_results[r]['entropy']['params']['approximate_entropy'] for r in records},
           "ApEn", threshold=1.5)

ax = fig.add_subplot(gs[3, 1])
styled_bar(ax, "Sample Entropy\n[Abnormal if > 1.2]",
           {r: all_results[r]['entropy']['params']['sample_entropy'] for r in records},
           "SampEn", threshold=1.2)

ax = fig.add_subplot(gs[3, 2])
styled_bar(ax, "Shannon Entropy\n[Higher = More Complex]",
           {r: all_results[r]['entropy']['params']['shannon_entropy'] for r in records},
           "Shannon Entropy")

# ── ROW 4: Voting Heatmap ──
ax_heat = fig.add_subplot(gs[4, :2])
vote_matrix = np.array([[1 if all_results[r][m]['abnormal'] else 0
                          for r in records] for m in methods])
im = ax_heat.imshow(vote_matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1)
ax_heat.set_xticks(range(len(records)))
ax_heat.set_xticklabels([f"R{r}\n{SHORT_LABELS[r][:13]}" for r in records], fontsize=9)
ax_heat.set_yticks(range(len(method_names)))
ax_heat.set_yticklabels(method_names, fontsize=9)
ax_heat.set_title("Voting Heatmap — All Methods × All Records\n(Green = Normal, Red = Abnormal)",
                   fontsize=11, fontweight='bold', pad=10)
for i in range(len(methods)):
    for j in range(len(records)):
        txt = "ABNORMAL" if vote_matrix[i, j] else "NORMAL"
        ax_heat.text(j, i, txt, ha='center', va='center',
                     fontsize=10, fontweight='bold', color='white')

# ── ROW 4: Diagnosis Summary ──
ax_diag = fig.add_subplot(gs[4, 2])
ax_diag.axis('off')
lines = ["  FINAL DIAGNOSIS\n  "]
for r in records:
    cnt = all_results[r]['abnormal_count']
    diag = all_results[r]['final_diagnosis']
    bar  = '█'*cnt + '░'*(5-cnt)
    lines += [f"  Record {r}: {SHORT_LABELS[r][:14]}",
              f"   Votes: {cnt}/5  {bar}",
              f"   ➜ {diag[:22]}", "  "]
ax_diag.text(0.04, 0.97, "\n".join(lines),
             transform=ax_diag.transAxes, fontsize=9, family='monospace',
             verticalalignment='top', color='white',
             bbox=dict(boxstyle='round', facecolor='#0f3460', alpha=0.75))
ax_diag.set_title("Diagnosis Summary", fontsize=10, fontweight='bold', pad=10)

# ── ROW 5: Legend ──
ax_leg = fig.add_subplot(gs[5, :])
ax_leg.axis('off')
patches = [
    mpatches.Patch(color='#2ecc71', label='Record 100 — Normal Sinus Rhythm'),
    mpatches.Patch(color='#e74c3c', label='Record 106 — Ventricular Arrhythmia'),
    mpatches.Patch(color='#e67e22', label='Record 207 — Atrial Fibrillation'),
    mpatches.Patch(color='yellow',  label='Clinical Threshold (dashed line)'),
]
ax_leg.legend(handles=patches, loc='center', ncol=4, fontsize=10,
              facecolor='#16213e', edgecolor='#444', labelcolor='white',
              framealpha=0.8)

fig.suptitle(
    "ECG Comprehensive Signal Processing Comparison Analysis\n"
    "Threshold-Based Arrhythmia Detection — No ML Required",
    fontsize=15, fontweight='bold', y=1.00
)

plt.savefig('ecg_comparison_analysis.png', dpi=200, bbox_inches='tight',
            facecolor=fig.get_facecolor())
print("✅ Figure saved as ecg_comparison_analysis.png")
plt.show()

---
## Cell 7 — RR Interval Histograms (Per Record)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#1a1a2e')

for ax, r in zip(axes, records):
    rr  = all_results[r]['rr']
    p   = all_results[r]['hist']['params']
    abn = all_results[r]['hist']['abnormal']
    ax.hist(rr, bins=30, color=COLORS[r], edgecolor='white', alpha=0.80)
    ax.axvline(p['mean_rr'], color='yellow', linestyle='--', linewidth=2,
               label=f"Mean: {p['mean_rr']:.0f} ms")
    status = '❌ ABNORMAL' if abn else '✅ NORMAL'
    ax.set_title(f"Record {r} — {SHORT_LABELS[r]}\n"
                 f"Std={p['std_rr']:.1f}ms  CV={p['cv_rr']:.3f}  {status}",
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('RR Interval (ms)', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.legend(fontsize=8, facecolor='#222', labelcolor='yellow')
    ax.grid(True, alpha=0.2)

fig.suptitle("RR Interval Histograms — Variability Comparison",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 8 — Wavelet Energy Distribution (Per Record)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#1a1a2e')
levels = ['A5', 'D5', 'D4', 'D3', 'D2', 'D1']

for ax, r in zip(axes, records):
    ep  = all_results[r]['wavelet']['params']['energy_distribution']
    de  = all_results[r]['wavelet']['params']['detail_energy']
    abn = all_results[r]['wavelet']['abnormal']
    bar_colors = ['#2ecc71' if e < 60 else '#e74c3c' for e in ep]
    ax.bar(levels, ep, color=bar_colors, edgecolor='white', linewidth=0.6)
    ax.axhline(60, color='yellow', linestyle='--', linewidth=1.5, label='60% threshold')
    status = '❌ ABNORMAL' if abn else '✅ NORMAL'
    ax.set_title(f"Record {r} — {SHORT_LABELS[r]}\n"
                 f"Detail Energy: {de:.1f}%  {status}",
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Decomposition Level', fontsize=8)
    ax.set_ylabel('Energy (%)', fontsize=8)
    ax.legend(fontsize=8, facecolor='#222', labelcolor='yellow')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle("Wavelet Energy Distribution — db4, 5 Levels",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 9 — STFT Spectrograms (Per Record)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.patch.set_facecolor('#1a1a2e')

for ax, r in zip(axes, records):
    f_stft = all_results[r]['f_stft']
    t_stft = all_results[r]['t_stft']
    Zxx    = all_results[r]['Zxx']
    sp     = all_results[r]['stft']['params']
    abn    = all_results[r]['stft']['abnormal']
    fmask  = f_stft <= 10
    im = ax.pcolormesh(t_stft, f_stft[fmask],
                       10*np.log10(np.abs(Zxx[fmask,:])**2+1e-12),
                       shading='gouraud', cmap='inferno')
    plt.colorbar(im, ax=ax, label='Power (dB)', shrink=0.8)
    status = '❌ ABNORMAL' if abn else '✅ NORMAL'
    ax.set_title(f"Record {r} — {SHORT_LABELS[r]}\n"
                 f"Stability CV={sp['freq_stability_cv']:.3f}  {status}",
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=8)
    ax.set_ylabel('Frequency (Hz)', fontsize=8)

fig.suptitle("STFT Spectrograms (0–10 Hz) — Temporal Frequency Stability",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 10 — FFT Power Spectra (Per Record)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#1a1a2e')

for ax, r in zip(axes, records):
    freqs  = all_results[r]['freqs']
    power  = all_results[r]['power']
    fp     = all_results[r]['fft']['params']
    abn    = all_results[r]['fft']['abnormal']
    cm     = (freqs >= 0.5) & (freqs <= 5)
    ax.plot(freqs[cm], power[cm], color=COLORS[r], linewidth=1.2)
    ax.axvline(fp['dominant_freq'], color='yellow', linestyle='--', linewidth=1.5,
               label=f"Dom: {fp['dominant_hr']:.1f} bpm")
    ax.axvspan(50/60, 120/60, alpha=0.1, color='lime')
    status = '❌ ABNORMAL' if abn else '✅ NORMAL'
    ax.set_title(f"Record {r} — {SHORT_LABELS[r]}\n"
                 f"Peaks={fp['num_peaks']}  HR={fp['dominant_hr']:.1f}bpm  {status}",
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Frequency (Hz)', fontsize=8)
    ax.set_ylabel('Power', fontsize=8)
    ax.legend(fontsize=8, facecolor='#222', labelcolor='yellow')
    ax.grid(True, alpha=0.2)

fig.suptitle("FFT Power Spectra (0.5–5 Hz) — Cardiac Frequency Analysis",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Cell 11 — Conclusion & Interpretation

In [ ]:
print("="*70)
print("  CONCLUSION: THRESHOLD-BASED ARRHYTHMIA DETECTION")
print("  (Pure Signal Processing — No Machine Learning)")
print("="*70)

for r in records:
    res  = all_results[r]
    cnt  = res['abnormal_count']
    diag = res['final_diagnosis']
    bar  = '█'*cnt + '░'*(5-cnt)

    print(f"\n  Record {r} — {res['label']}")
    print(f"  {'─'*50}")
    for m, mn in zip(methods, method_names):
        abn   = res[m]['abnormal']
        flags = res[m]['flags']
        hit   = [k for k, v in flags.items() if v]
        icon  = '❌' if abn else '✅'
        detail = f"  → {', '.join(hit)}" if hit else ""
        print(f"  {icon} {mn:<12} {'ABNORMAL' if abn else 'NORMAL  '}{detail}")
    print(f"  {'─'*50}")
    print(f"  Votes: {cnt}/5  {bar}")
    print(f"  FINAL: {diag}")

print(f"\n{'='*70}")
print("  KEY TAKEAWAY")
print("  By comparing signal parameters against clinical thresholds,")
print("  we can detect arrhythmia patterns WITHOUT any machine learning.")
print("  The voting system ensures robustness — no single method")
print("  alone determines the diagnosis.")
print("="*70)